# Chapter 6. Mixture Models

In [ ]:
import os
import warnings

import arviz as az
import matplotlib.pyplot as plt
import pandas as pd

import jax.numpy as jnp
from jax import random, local_device_count

import numpyro
import numpyro.distributions as dist

from numpyro.infer import MCMC, NUTS, Predictive
from numpyro.distributions.transforms import OrderedTransform

seed=1234

if "SVG" in os.environ:
    %config InlineBackend.figure_formats = ["svg"]
warnings.formatwarning = lambda message, category, *args, **kwargs: "{}: {}\n".format(
    category.__name__, message
)
az.style.use("arviz-darkgrid")
numpyro.set_platform("cpu") # or "gpu", "tpu" depending on system
numpyro.set_host_device_count(local_device_count())

In [ ]:
# import pymc3 as pm
# import numpy as np
# import scipy.stats as stats
# import pandas as pd
# import theano.tensor as tt
# import matplotlib.pyplot as plt
# import arviz as az

In [ ]:
# az.style.use('arviz-darkgrid')

In [ ]:
# np.random.seed(42)

In [ ]:
cs = pd.read_csv('../data/chemical_shifts_theo_exp.csv')
cs_exp = cs['exp']
az.plot_kde(cs_exp)
plt.hist(cs_exp, density=True, bins=30, alpha=0.3)
plt.yticks([])

In [ ]:
#with pm.Model() as model_kg:
#    p = pm.Dirichlet('p', a=np.ones(clusters))
#    z = pm.Categorical('z', p=p, shape=len(cs_exp))
#    means = pm.Normal('means', mu=cs_exp.mean(), sd=10, shape=clusters)
#    sd = pm.HalfNormal('sd', sd=10)
#
#    y = pm.Normal('y', mu=means[z], sd=sd, observed=cs_exp)
#    trace_kg = pm.sample()

In [ ]:
# with numpyro.handlers.seed(rng_seed=seed):
#     N = 1  # Samples
#     b = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(clusters)).expand([N]))
# # dist.Dirichlet(concentration=jnp.ones(clusters)).sample(random.PRNGKey(0), (1,))

In [ ]:
# b.squeeze()

In [ ]:
clusters = 2

def model(obs=None):
    p = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(clusters)))
    c = dist.Categorical(probs=p.squeeze())

    means = numpyro.sample('means', dist.Normal(loc=cs_exp.mean(), scale=10), sample_shape=(clusters,))
    sd = numpyro.sample('sd', dist.HalfNormal(scale=10))
    component_dist = dist.Normal(loc=means, scale=sd)
    
    y = numpyro.sample('y', dist.MixtureSameFamily(mixing_distribution=c, component_distribution=component_dist), obs=obs)
    
kernel = NUTS(model)
mcmc = MCMC(kernel, num_warmup=500, num_samples=2000, num_chains=2, chain_method='sequential')
mcmc.run(random.PRNGKey(seed), obs=jnp.asarray(cs_exp))

In [ ]:
mcmc.get_samples()

In [ ]:
varnames = ['means', 'p']
az.plot_trace(mcmc, varnames, compact=False)

In [ ]:
# clusters = 2
# with pm.Model() as model_mg:
#     p = pm.Dirichlet('p', a=np.ones(clusters))
#     means = pm.Normal('means', mu=cs_exp.mean(), sd=10, shape=clusters)
#     sd = pm.HalfNormal('sd', sd=10)
#     y = pm.NormalMixture('y', w=p, mu=means, sd=sd, observed=cs_exp)
#     trace_mg = pm.sample(random_seed=123)

In [ ]:
# varnames = ['means', 'p']
# az.plot_trace(trace_mg, varnames)
# plt.savefig('B11197_06_06.png')

In [ ]:
az.summary(mcmc, var_names=varnames)

In [ ]:
# az.summary(trace_mg, varnames)

## Non-identifiability of mixture models

In [ ]:
jnp.array([.9, 1]) * cs_exp.mean()

In [ ]:
jnp.expand_dims(jnp.asarray(cs_exp), axis=1).shape

In [ ]:
clusters = 2

def model(obs=None):
    p = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(clusters)))
    c = dist.Categorical(probs=p.squeeze())
    
    mu = jnp.array([.9, 1]) * cs_exp.mean()

    means = numpyro.sample('means', dist.Normal(loc=mu, scale=10), sample_shape=(2,))
    sd = numpyro.sample('sd', dist.HalfNormal(scale=10))
    component_dist = dist.Normal(loc=means, scale=sd)
    
    y = numpyro.sample('y', dist.MixtureSameFamily(mixing_distribution=c, component_distribution=component_dist), obs=obs)
    
kernel = NUTS(model)
mcmc2 = MCMC(kernel, num_warmup=500, num_samples=2000, num_chains=2, chain_method='sequential')
mcmc2.run(random.PRNGKey(seed), obs=jnp.expand_dims(jnp.asarray(cs_exp), axis=1))

In [ ]:
varnames = ['means', 'p']
az.plot_trace(mcmc2, varnames, compact=False)

In [ ]:
# instead of a potential we can use an ordered transformation
# transform=pm.distributions.transforms.ordered

In [ ]:
az.summary(mcmc2)

## How to choose K

In [ ]:
def model(obs=None):
    p = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(clusters)))
    c = dist.Categorical(probs=p.squeeze())
    
    mu = jnp.array([.9, 1]) * cs_exp.mean()

    means = numpyro.sample('means', dist.Normal(loc=mu, scale=10), sample_shape=(2,))
    sd = numpyro.sample('sd', dist.HalfNormal(scale=10))
    component_dist = dist.Normal(loc=means, scale=sd)
    
    y = numpyro.sample('y', dist.MixtureSameFamily(mixing_distribution=c, component_distribution=component_dist), obs=obs)
    
kernel = NUTS(model)
mcmc2 = MCMC(kernel, num_warmup=500, num_samples=2000, num_chains=2, chain_method='sequential')
mcmc2.run(random.PRNGKey(seed), obs=jnp.expand_dims(jnp.asarray(cs_exp), axis=1))

In [ ]:
clusters = [3, 4, 5, 6]

models = []
traces = []
for cluster in clusters:
    def model(obs=None):
        
        p = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(cluster)))
        c = dist.Categorical(probs=p.squeeze())
    
        mu = jnp.linspace(cs_exp.min(), cs_exp.max(), cluster)
        means = numpyro.sample('means', 
                               dist.TransformedDistribution(
                                   base_distribution=dist.Normal(loc=mu, scale=10).expand([cluster]), 
                               transforms=OrderedTransform()
                              ))
        print(means)
        sd = numpyro.sample('sd', dist.HalfNormal(scale=10))
        component_dist = dist.Normal(loc=means, scale=sd)
        print(c.probs.shape)
        y = numpyro.sample('y', dist.MixtureSameFamily(mixing_distribution=c, component_distribution=component_dist), obs=obs)
        
    kernel = NUTS(model)
    trace = MCMC(kernel, num_warmup=500, num_samples=2000, num_chains=2, chain_method='sequential')
    trace.run(random.PRNGKey(seed), obs=jnp.expand_dims(jnp.asarray(cs_exp), axis=1))
    traces.append(trace)
    models.append(model)

In [ ]:
_, ax = plt.subplots(2, 2, figsize=(11, 8), constrained_layout=True)
 
ax = list(ax.flat)
x = jnp.linspace(cs_exp.min(), cs_exp.max(), 200)
for idx, trace_x in enumerate(traces):
    x_ = jnp.array([x] * clusters[idx]).T
 
    for i in range(50):
        i_ = int(dist.Uniform(low=0, high=len(trace_x.get_samples()['means'])).sample(key=random.PRNGKey(i)))
        means_y = trace_x.get_samples()['means'][i_]
        p_y = trace_x.get_samples()['p'][i_]
        sd = trace_x.get_samples()['sd'][i_]
        distri = dist.Normal(loc=means_y, scale=sd)
        ax[idx].plot(x, jnp.sum(jnp.exp(distri.log_prob(x_)) * p_y, 1), 'C0', alpha=0.1)
 
    means_y = trace_x.get_samples()['means'].mean(0)
    p_y = trace_x.get_samples()['p'].mean(0)
    sd = trace_x.get_samples()['sd'].mean()
    distri = dist.Normal(loc=means_y, scale=sd)
    #stats.norm(means_y, sd)
    ax[idx].plot(x, jnp.sum(jnp.exp(distri.log_prob(x_)) * p_y, 1), 'C0', lw=2)
    ax[idx].plot(x, jnp.exp(distri.log_prob(x_)) * p_y, 'k--', alpha=0.7)
         
    az.plot_kde(cs_exp, plot_kwargs={'linewidth':2, 'color':'k'}, ax=ax[idx])
    ax[idx].set_title('K = {}'.format(clusters[idx]))
    ax[idx].set_yticks([])
    ax[idx].set_xlabel('x')

In [ ]:
# prior = Predictive(mcmc_l.sampler.model, num_samples=10)
# prior_p = prior(random.PRNGKey(seed), obs=y_1s)



In [ ]:
Predictive(model=traces[0].sampler.model, 
                     posterior_samples=traces[0].get_samples(), 
                     return_sites=['y'])(random.PRNGKey(seed)).values()

In [ ]:
ppc_mm = [
            Predictive(model=traces[i].sampler.model, 
                     posterior_samples=traces[i].get_samples(), 
                     return_sites=['y'])(random.PRNGKey(seed))
          for i in range(4)]

In [ ]:
type(ppc_mm)

In [ ]:
for idx, d_sim in enumerate(list(ppc_mm)):
    print(idx, d_sim['y'])

In [ ]:
jnp.expand_dims(d_sim['y'][:100].T, axis=1).shape

In [ ]:
# iqr(d_sim['y'][:100].T, 0)

In [ ]:
# ppc_mm = [pm.sample_posterior_predictive(traces[i], 1000, models[i])
#           for i in range(4)]

In [ ]:
fig, ax = plt.subplots(2, 2, figsize=(10, 6), sharex=True, constrained_layout=True)
ax = list(ax.flat)
def iqr(x, a=0):
    return jnp.subtract(*jnp.percentile(jnp.asarray(x), jnp.array([75, 25]), axis=a))

T_obs = iqr(cs_exp)
for idx, d_sim in enumerate(ppc_mm):
    ds = jnp.expand_dims(d_sim['y'][:100], axis=1)
    T_sim = iqr(ds.T, 0)
    print(T_sim)
    p_value = jnp.mean(T_sim >= T_obs)
    az.plot_kde(T_sim, ax=ax[idx])
    ax[idx].axvline(T_obs, 0, 1, color='k', ls='--')
    ax[idx].set_title(f'K = {clusters[idx]} \n p-value {p_value:.2f}')
    ax[idx].set_yticks([])

In [ ]:
idata_list = [az.from_numpyro(t) for t in traces]
comp = az.compare(dict(zip(clusters, idata_list)), ic="waic", method='BB-pseudo-BMA')
comp

In [ ]:
az.plot_compare(comp)

## Non-finite mixture model

In [ ]:
def stick_breaking_truncated(α, H, K):
    """
    Truncated stick-breaking process view of a DP
    
    Parameters
    ----------
    α : float
        concentration parameter
    H : `numpyro` distribution
        base distribution
    K : int
        number of components
    
    Returns
    -------
    locs : array
        locations
    w : array
        probabilities
    """
    
#     βs = stats.beta.rvs(1, α, size=K)
    βs = dist.Beta(concentration1=1, concentration0=α).sample(random.PRNGKey(1), (K,))
    w = jnp.empty(K)
    w = βs * jnp.concatenate((jnp.array([1.]), jnp.cumprod(1 - βs[:-1])))
    locs = H.sample(random.PRNGKey(1), (K,))
    return locs, w

# Parameters DP
K = 500
H = dist.Normal()
alphas = [1, 10, 100, 1000]

# plot
_, ax = plt.subplots(2, 2, sharex=True, figsize=(10, 5))
ax = list(ax.flat)
for idx, α in enumerate(alphas):
    locs, w = stick_breaking_truncated(α, H, K)
    ax[idx].vlines(locs, 0, w, color='C0')
    ax[idx].set_title('α = {}'.format(α))

plt.tight_layout()

In [ ]:
α = 10
H = dist.Normal()
K = 5

x = jnp.linspace(-4, 4, 250)
x_ = jnp.array([x] * K).T
locs, w = stick_breaking_truncated(α, H, K)

# dist = stats.laplace(locs, 0.5)
distri = dist.Laplace(loc=locs, scale=0.5)
plt.plot(x, jnp.sum(jnp.exp(distri.log_prob(x_)) * w, 1), 'C0', lw=2)
plt.plot(x, jnp.exp(distri.log_prob(x_)) * w, 'k--', alpha=0.7)
plt.yticks([])

In [ ]:
K = 20

def stick_breaking(α, K):
    β = numpyro.sample('β', dist.Beta(concentration1=1., concentration0=α), sample_shape=(K,))
    w = β * jnp.concatenate([jnp.array([1.]), jnp.cumprod(1. - β)[:-1]])
#     β = pm.Beta('β', 1., α, shape=K)
#     w = β * pm.math.concatenate([[1.], tt.extra_ops.cumprod(1. - β)[:-1]])

    return w

In [ ]:
def model(obs=None):
    α = numpyro.sample('α', dist.Gamma(concentration=1, rate=1.))
    w = numpyro.deterministic('w', stick_breaking(α, K))
                       
#     p = numpyro.sample("p", dist.Dirichlet(concentration=jnp.ones(clusters)))
    c = dist.Categorical(probs=w.squeeze())
    
    mu = jnp.linspace(cs_exp.min(), cs_exp.max(), K)

    means = numpyro.sample('means', dist.Normal(loc=mu, scale=10), sample_shape=(K,))
                       
    sd = numpyro.sample('sd', dist.HalfNormal(scale=10))
                       
    component_dist = dist.Normal(loc=means, scale=sd)
    
    obss = numpyro.sample('obss', dist.MixtureSameFamily(mixing_distribution=c, component_distribution=component_dist), obs=obs)
    
kernel = NUTS(model, target_accept_prob=0.85)
mcmc3 = MCMC(kernel, num_warmup=50, num_samples=50, num_chains=2, chain_method='sequential')
mcmc3.run(random.PRNGKey(seed), obs=jnp.expand_dims(jnp.asarray(cs_exp.values), axis=1))

In [ ]:
# with pm.Model() as model:
#     α = pm.Gamma('α', 1, 1.)
#     w = pm.Deterministic('w', stick_breaking(α, K))
#     means = pm.Normal('means',
#                       mu=np.linspace(cs_exp.min(), cs_exp.max(), K),
#                       sd=10, shape=K)
    
#     sd = pm.HalfNormal('sd', sd=10, shape=K)
#     obs = pm.NormalMixture('obs', w, means, sd=sd, observed=cs_exp.values)
#     trace = pm.sample(1000, tune=2000, nuts_kwargs={'target_accept':0.85})

In [ ]:
az.plot_trace(mcmc3, var_names=['α'], divergences=False, compact=False);

In [ ]:
az.plot_trace(mcmc3, var_names=['α'], divergences=False);

In [ ]:
plt.figure(figsize=(8, 6))
plot_w = jnp.arange(K)
plt.plot(plot_w, mcmc3.get_samples()['w'].mean(0), 'o-')
plt.xticks(plot_w, plot_w+1)
plt.xlabel('Component')
plt.ylabel('Average weight')

In [ ]:
traces[0].get_samples()['means']

In [ ]:
mcmc3.get_samples()['means'].shape

In [ ]:
jnp.expand_dims(mcmc3.get_samples()['means'], axis=0).shape

In [ ]:
mcmc3.get_samples()['means'][:, jnp.newaxis, :].shape

In [ ]:
mcmc3.get_samples()['sd'].shape, jnp.expand_dims(mcmc3.get_samples()['sd'], axis=1).shape

In [ ]:
mcmc3.get_samples()['w'][:, jnp.newaxis, :].shape, jnp.expand_dims(mcmc3.get_samples()['w'], axis=2).shape

In [ ]:
post = mcmc3.get_samples()
w = post['w']                                             # (S, K)
_means_raw = post['means']                                # (S, K, K) due to sample_shape=(K,)
means = _means_raw[:, 0, :]                               # (S, K) - first draw per posterior sample
sd = post['sd'][:, None]                                  # (S, 1) -> broadcasts to (S, K)

x_plot = jnp.linspace(cs_exp.min(), cs_exp.max(), 200)   # (G,)
comp_dens = jnp.exp(
    dist.Normal(means[..., None], sd[..., None]).log_prob(x_plot)
)                                                         # (S, K, G)
post_pdfs = jnp.sum(w[..., None] * comp_dens, axis=1)    # (S, G)

In [ ]:
plt.figure(figsize=(8, 6))

plt.hist(cs_exp.values, bins=25, density=True, alpha=0.5)
plt.plot(x_plot, post_pdfs[::100].T, c='0.5')
plt.plot(x_plot, post_pdfs.mean(axis=0), c='k')

plt.xlabel('x')
plt.yticks([])

In [ ]:
## Exercises
# clusters = 3
# n_cluster = [200, 150, 170]
# n_total = sum(n_cluster)
# means = [5, 0, -3]
# std_devs = [2, 2, 2]
# mix = np.random.normal(jnp.repeat(means, n_cluster),
# np.repeat(std_devs, n_cluster))
# az.plot_kde(np.array(mix));